In [ ]:
import requests
import string
import json
import time
import os
import zipfile
import io
import shutil
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

tqdm.pandas()

from dotenv import load_dotenv

load_dotenv()


In [ ]:
USE_CACHED_REPOS = True

OPTED_OUT_REPO_IDS = []


In [ ]:
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

GITHUB_REPOS_DIR = CACHE_DIR / "github_repos"
GITHUB_REPOS_DIR.mkdir(parents=True, exist_ok=True)

# Create a temporary directory for processing
TEMP_DIR = CACHE_DIR / "temp"
TEMP_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
from requests.adapters import HTTPAdapter, Retry

req_session = requests.Session()

retries = Retry(total=3, backoff_factor=0.1, status_forcelist=[500, 502, 503, 504])

req_session.mount("https://", HTTPAdapter(max_retries=retries))

# Headers for API request
headers = {
    "Accept": "application/vnd.github.v3+json",
    "Authorization": f"token {os.getenv('GITHUB_TOKEN')}",
}


In [ ]:
# List to store all repositories
def get_all_repos():
    cached_file = GITHUB_REPOS_DIR / "all_repos.json"
    if USE_CACHED_REPOS and cached_file.exists():
        return json.loads(cached_file.read_text(encoding="utf-8"))

    all_repos = []

    pbar = tqdm(
        desc=f"Finding Lua/Luau repositories with permissive licenses",
        bar_format="{desc}: {n_fmt} [{elapsed}{postfix}]",
    )

    queries: list[tuple[str, str, str, int]] = []

    # Search through each letter of the alphabet and some common terms
    search_terms = [
        "rbx",
        "rblx",
        "roblox",
        "luau",
        "lua",
    ] + [letter for letter in string.ascii_lowercase]
    # Filter for permissive licenses
    licenses = [
        "MIT",
        "BSL-1.0",
        "BSD-2-Clause",
        "BSD-3-Clause",
        "0BSD",
        "CC0-1.0",
        "CC-BY-4.0",
        "CC-BY-SA-4.0",
        "WTFPL",
        "ISC",
        "MS-PL",
        "Unlicense",
        "Zlib",
    ]
    for search_term in search_terms:
        for license in licenses:
            queries.append((search_term, "luau", license, 1))
            queries.append((search_term + "%20NOT%20nvim", "lua", license, 1))

    for query, language, license, page in queries:
        url = f"https://api.github.com/search/repositories?q={query}+language:{language}+license:{license}+fork:false&sort=stars&order=desc&page={page}&per_page=100"
        pbar.set_postfix_str(
            f"Query {query}, Language {language}, License {license}, Page {page}"
        )
        time.sleep(1.0 / 30.0)  # Space out requests to avoid hitting rate limits
        try:
            response = req_session.get(url, headers=headers)

            remaining = int(response.headers.get("X-RateLimit-Remaining", "1"))
            if remaining == 0:
                reset_utc = int(response.headers.get("X-RateLimit-Reset", "0"))
                reset_delay = max(2, reset_utc - int(time.time()))

                tqdm.write(f"Rate limit exceeded, pausing for {reset_delay} seconds...")
                time.sleep(reset_delay)
                # Retry this request
                queries.append((query, language, license, page))
                continue

            if response.status_code != 200:
                tqdm.write(f"Error: {response.status_code} - {response.text}")
                continue

            data = response.json()
            items = data.get("items", [])

            if not items:
                continue

            all_repos.extend(items)
            pbar.update(len(items))

            if len(items) == 100 and page < 10:
                # Get the next page
                queries.append((query, language, license, page + 1))
        except Exception as e:
            tqdm.write(f"Error: {e.__class__.__name__} {str(e)}")

    pbar.close()

    unique_repos = []
    seen_ids = set()
    for repo in all_repos:
        if repo["id"] in seen_ids:
            continue
        seen_ids.add(repo["id"])
        unique_repos.append(repo)

    # Sort repositories by star count
    unique_repos.sort(key=lambda x: x["stargazers_count"], reverse=True)
    tqdm.write(f"Found {len(unique_repos)} possible repositories")

    cached_file.write_text(json.dumps(unique_repos, indent=2), encoding="utf-8")

    return unique_repos


In [ ]:
def is_repo_luau(repo):
    is_luau = (repo["language"] or "").lower() == "luau"
    if is_luau:
        return True

    lower_name = repo["full_name"].lower()
    lower_description = (repo["description"] or "").lower()

    # If the repo is roblox related, include it
    if (
        "roblox" in lower_name
        or "roblox" in lower_description
        or "rbx" in lower_name
        or "rbx" in lower_description
        or "rblx" in lower_name
        or "rblx" in lower_description
    ):
        return True

    # If the repo is neovim related, drop it
    if "vim" in lower_name or "vim" in lower_description:
        return False

    # This Lua repo might not be Luau. Let's see if it's got some files that Luau repos often do.
    for file_name in [
        "default.project.json",
        "selene.toml",
        "wally.toml",
        "stylua.toml",
    ]:
        file_url = f"https://raw.githubusercontent.com/{repo['full_name']}/refs/heads/{repo['default_branch']}/{file_name}"
        # If we get a 200, this file exists and the repo is likely Luau
        try:
            time.sleep(1.0 / 80.0)  # Be gentle(ish) with the API
            response = req_session.get(file_url, headers=headers)
            if response.status_code == 200:
                return True
        except Exception as e:
            tqdm.write(f"Error checking {file_url}: {e}")

    return False


In [ ]:
def get_luau_repos():
    cached_file = GITHUB_REPOS_DIR / "luau_repos.json"
    if USE_CACHED_REPOS and cached_file.exists():
        return json.loads(cached_file.read_text(encoding="utf-8"))

    repos = get_all_repos()

    luau_repos = []
    for repo in tqdm(repos, desc="Filtering to likely Luau repositories"):
        if is_repo_luau(repo):
            luau_repos.append(repo)

    tqdm.write(f"Filtered to {len(luau_repos)} likely Luau repositories")

    cached_file.write_text(json.dumps(luau_repos, indent=2), encoding="utf-8")

    return luau_repos


In [ ]:
df = pd.DataFrame(
    columns=[
        "file_path",
        "file_url",
        "repo_name",
        "repo_description",
        "contributors",
        "commit_hash",
        "license",
        "license_url",
        "file_content",
    ]
)


In [ ]:
import subprocess


def download_repo(repo):
    if repo["id"] in OPTED_OUT_REPO_IDS:
        tqdm.write(f"Skipping {repo['full_name']}, opted out")
        return

    repo_name: str = repo["full_name"]
    default_branch: str = repo["default_branch"] or "master"

    zip_url: str = (
        f"https://github.com/{repo_name}/archive/refs/heads/{default_branch}.zip"
    )

    # Create a temporary directory for this repo
    repo_dir: Path = TEMP_DIR / repo_name.replace("/", "_")
    repo_dir.mkdir(parents=True, exist_ok=True)
    try:
        # Try to download the zip file
        response = req_session.get(zip_url, headers=headers)
        if response.status_code != 200:
            tqdm.write(f"Failed to download {repo_name}: {response.status_code}")
            repo_dir.rmdir()  # Clean up the empty directory
            return

        # Extract the files we need
        with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
            namelist = zip_ref.namelist()

            if not any(
                [
                    name.lower().endswith(
                        (
                            ".lua",
                            ".luau",
                        )
                    )
                    for name in namelist
                ]
            ):
                tqdm.write(f"Skipping {repo_name}, no code files found")
                return

            for name in namelist:
                if name.lower().endswith(
                    (
                        ".lua",
                        ".luau",
                        "license",
                        "license.md",
                        "license.txt",
                        "stylua.toml",
                    )
                ) and not name.lower().endswith( # Skip type definition files
                    ".d.luau"
                ):
                    zip_ref.extract(name, repo_dir)
                    # tqdm.write(f"Extracted {repo_name}/{name}")

        # The first dir inside the repo is gonna be repo_name-branch_name.
        # Let's move everything up into the repo dir itself.
        first_dir = next(repo_dir.iterdir())
        # Move everything in first_dir to repo_dir
        for item in first_dir.iterdir():
            shutil.move(item, repo_dir)
        # Remove the first dir
        shutil.rmtree(first_dir, ignore_errors=True)

        return repo_dir
    except Exception as e:
        tqdm.write(f"Error downloading {repo_name}: {e}")
        shutil.rmtree(repo_dir, ignore_errors=True)
        return None


def ingest_repo(repo, repo_dir: Path):
    repo_name: str = repo["full_name"]
    default_branch: str = repo["default_branch"] or "master"
    repo_url: str = f"https://github.com/{repo_name}"

    # Gather the contributors
    contributors = []
    try:
        contributors_response = req_session.get(
            f"https://api.github.com/repos/{repo_name}/contributors?per_page=100",
            headers=headers,
        )
        if contributors_response.status_code == 200:
            contributors = [c["login"] for c in contributors_response.json()]
    except Exception as e:
        tqdm.write(f"Error gathering contributors for {repo_name}: {e}")

    # Get the commit hash
    commit_url = f"https://api.github.com/repos/{repo_name}/commits/{default_branch}"
    commit_response = req_session.get(commit_url, headers=headers)
    commit_hash = (
        commit_response.json()["sha"]
        if commit_response.status_code == 200
        else default_branch
    )

    # Find license file
    license_content = ""
    license_path = ""
    license_files = []

    for license_name in [
        "LICENSE",
        "LICENSE.md",
        "LICENSE.txt",
        "license",
        "license.md",
        "license.txt",
    ]:
        found_licenses = list(repo_dir.glob(f"**/{license_name}"))
        license_files.extend(found_licenses)

    if license_files:
        license_path = str(license_files[0].relative_to(repo_dir))
        with open(license_files[0], "r", encoding="utf-8", errors="ignore") as f:
            license_content = f.read()

    # Generate license URL
    license_url = (
        f"{repo_url}/blob/{commit_hash}/{license_path}" if license_path else ""
    )

    # Find all Lua/Luau files
    lua_files = list(repo_dir.glob("**/*.lua"))
    luau_files = list(repo_dir.glob("**/*.luau"))
    all_files = lua_files + luau_files

    # Process each file
    added_files = 0
    for file_path in all_files:
        rel_path = file_path.relative_to(repo_dir)
        try:
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                file_content = f.read()

            if file_content.strip() == "" or len(file_content) > 50000 or file_content.count("\n") > 3000:
                continue

            if any(banned_keyword in file_content.lower() for banned_keyword in [
                # Cursed codegen from WASM
                "wasynth",
                "wasm",
                # Luau removed the io library
                "io.read(",
                "io.write(",
                "io.line(",
                # Keep bad actor's code out of this dataset
                "exploit",
                "condo",
                "sex",
                "porn",
                "r34",
            ]):
                continue

            process = subprocess.Popen(
                [
                    "stylua",
                    "--syntax",
                    "luau",
                    "-"
                ],
                cwd=str(repo_dir.resolve()),
                stdin=subprocess.PIPE,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                encoding="utf-8",
            )
            try:
                output, err = process.communicate(timeout=20, input=file_content)
                if len(err) > 0:
                    continue
                elif len(output) > 0:
                    file_content = output
            except Exception as e:
                tqdm.write(f"\033[31mStylua failed to run:\033[0m {e}")
                continue

            # Add to DataFrame
            df.loc[len(df)] = {
                "file_path": str(rel_path),
                "file_url": f"{repo_url}/blob/{commit_hash}/{rel_path}",
                "repo_name": repo_name,
                "repo_description": repo["description"],
                "contributors": contributors,
                "commit_hash": commit_hash,
                "license": license_content,
                "license_url": license_url,
                "file_content": file_content,
            }
            added_files += 1
        except Exception as e:
            tqdm.write(f"Error processing {file_path}: {e.__class__.__name__} {str(e)}")

    tqdm.write(f"Added {added_files} files from {repo_name}")
    # Clean up
    shutil.rmtree(repo_dir, ignore_errors=True)


In [ ]:
luau_repos = get_luau_repos()
pbar = tqdm(luau_repos, desc="Processing repos")
for repo in pbar:
    pbar.set_description(
        f"Processing {repo['full_name']} ({repo['language'] or 'unknown'})"
    )

    time.sleep(1.0 / 30.0)  # Be gentle with the API

    repo_dir = download_repo(repo)
    if not repo_dir:
        tqdm.write(f"Failed to download {repo['full_name']}")
        continue

    ingest_repo(repo, repo_dir)

tqdm.write(f"Processed {len(df)} Lua/Luau files from {len(luau_repos)} repositories")



In [ ]:
df.to_parquet("train.parquet")
df
